<a href="https://colab.research.google.com/github/VitorEduardoOliveira/NLP/blob/main/ChatBotCSV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio sentence-transformers numpy pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 53.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files

# ── Upload do arquivo ──────────────────────────────────────────────────────────
print("Selecione seu arquivo CSV:")
uploaded = files.upload()
nome_arquivo = list(uploaded.keys())[0]

# ── Leitura e validação ────────────────────────────────────────────────────────
df = pd.read_csv(nome_arquivo)

assert "pergunta" in df.columns and "resposta" in df.columns, \
    "O CSV precisa ter as colunas 'pergunta' e 'resposta'"

df = df.dropna(subset=["pergunta", "resposta"])
df["pergunta"] = df["pergunta"].str.strip().str.lower()
df["resposta"] = df["resposta"].str.strip()

perguntas = df["pergunta"].tolist()
respostas = df["resposta"].tolist()

print("CSV carregado: " + str(len(perguntas)) + " entradas encontradas.")
print(df.head())

# ── Modelo + embeddings ────────────────────────────────────────────────────────
modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Gerando embeddings...")
embeddings_base = modelo.encode(perguntas, convert_to_numpy=True, show_progress_bar=True)
print("Pronto!")

Selecione seu arquivo CSV:


Saving base_conhecimento.csv to base_conhecimento.csv
CSV carregado: 30 entradas encontradas.
                 pergunta                                           resposta
0          o que e python  Python e uma linguagem de programacao de alto ...
1      o que e javascript  JavaScript e uma linguagem de programacao usad...
2         o que e uma api  API (Application Programming Interface) e um c...
3  o que e banco de dados  Banco de dados e um sistema organizado para ar...
4             o que e git  Git e um sistema de controle de versao que per...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Gerando embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Pronto!


In [ ]:
def responder(pergunta_usuario: str, limiar: float = 0.35, top_k: int = 1) -> str:
    if not pergunta_usuario.strip():
        return "Por favor, digite uma pergunta."

    embedding_usuario = modelo.encode([pergunta_usuario], convert_to_numpy=True)
    similaridades     = cosine_similarity(embedding_usuario, embeddings_base).flatten()

    # Pega os top_k resultados mais similares
    indices_top = np.argsort(similaridades)[::-1][:top_k]
    melhor_score = float(similaridades[indices_top[0]])

    if melhor_score < limiar:
        return "Nao encontrei informacao relevante no documento (score: " + f"{melhor_score:.2f}" + "). Tente reformular."

    if top_k == 1:
        return respostas[indices_top[0]] + "\n\nSimilaridade: " + f"{melhor_score:.2f}"

    # top_k > 1: retorna múltiplos trechos (útil para PDFs longos)
    partes = []
    for i, idx in enumerate(indices_top):
        score = float(similaridades[idx])
        partes.append(f"[Trecho {i+1} — score {score:.2f}]\n{respostas[idx]}")
    return "\n\n---\n\n".join(partes)

In [ ]:
import gradio as gr

with gr.Blocks(title="Chatbot Semantico", theme=gr.themes.Soft()) as demo:

    gr.Markdown("# Chatbot Semantico\nBase de conhecimento carregada de arquivo CSV ou PDF.")

    chatbot = gr.Chatbot(label="Conversa", height=440, bubble_full_width=False)

    with gr.Row():
        campo = gr.Textbox(
            placeholder="Digite sua pergunta...",
            label="",
            scale=5,
            autofocus=True,
        )
        btn = gr.Button("Enviar", variant="primary", scale=1)

    with gr.Accordion("Configuracoes avancadas", open=False):
        limiar_slider = gr.Slider(
            minimum=0.1, maximum=0.95, value=0.35, step=0.05,
            label="Limiar de similaridade",
            info="Abaixo desse valor o bot indica que nao encontrou resposta"
        )
        topk_slider = gr.Slider(
            minimum=1, maximum=5, value=1, step=1,
            label="Numero de trechos retornados (top-k)",
            info="Util para PDFs: retorna mais de um trecho relevante"
        )

    def turno(mensagem, historico, limiar, top_k):
        if not mensagem.strip():
            return historico, ""
        resposta = responder(mensagem, limiar, int(top_k))
        historico = historico + [(mensagem, resposta)]
        return historico, ""

    btn.click(fn=turno, inputs=[campo, chatbot, limiar_slider, topk_slider], outputs=[chatbot, campo])
    campo.submit(fn=turno, inputs=[campo, chatbot, limiar_slider, topk_slider], outputs=[chatbot, campo])

demo.launch(share=True)

/tmp/ipykernel_16444/3590482595.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Chatbot Semantico", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_16444/3590482595.py:7: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Conversa", height=440, bubble_full_width=False)
/tmp/ipykernel_16444/3590482595.py:7: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(label="Conversa", height=440, bubble_full_width=False)
/tmp/ipykernel_16444/3590482595.py:7: DeprecationWarning: The def

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a604dc32f3be095966.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
